# Phase Objective

#### Fine-tune the Decision Tree and Naive Bayes classifiers for improved performance using hyperparameter tuning techniques.

* Instructions

1. Understand the significance of hyperparameter tuning
2. Implement hyperparameter tuning using Python and evaluate the performance of the models after tuning
3. Analyze the results and compare the performance of the tuned models with the initial models

#### Implementing Hyperparameter Tuning on Our Classifiers
Now that we understand the importance of hyperparameter tuning, we will apply it to our Decision Tree and Naïve Bayes classifiers to optimize their performance.
##### Key Hyperparameters in NLP Classifiers
Here are some important hyperparameters for Decision Tree and Naïve Bayes classifiers:

* Decision Tree:
max_depth: Limits tree depth to avoid overfitting.
min_samples_split: Controls when a node should split to form more branches.
criterion: Determines how splits are chosen.

* Naive Bayes:
alpha (Laplace smoothing): Prevents zero probabilities for unseen words.
fit_prior: Determines if the model should learn class probabilities from data.

In [19]:
#Import our Libaries

import pandas as pd
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /Users/user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/user/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [33]:
import pandas as pd

# Load dataset
df = pd.read_csv("Bitext_sample_suportTraining.csv")


df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [35]:
df = df[['instruction', 'intent']]
df.dropna(inplace=True)

In [37]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\W', ' ', text)  # Remove punctuation
    tokens = word_tokenize(text)  # Tokenization
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]  # Lemmatization & stopword removal
    return " ".join(tokens)

df["cleaned_instruction"] = df["instruction"].apply(preprocess_text)

[nltk_data] Downloading package punkt to /Users/user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/user/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Convert text into numerical vectors
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df["cleaned_instruction"])
y = df["intent"]  # Target variable

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [88]:
dt_initial = DecisionTreeClassifier(random_state=42)
dt_initial.fit(X_train, y_train)
y_pred_dt_initial = dt_initial.predict(X_test)

In [90]:
nb_initial = MultinomialNB()
nb_initial.fit(X_train, y_train)
y_pred_nb_initial = nb_initial.predict(X_test)

In [94]:
print("\nInitial Decision Tree Performance:\n", classification_report(y_test, y_pred_dt_initial))
print("\nInitial Naïve Bayes Performance:\n", classification_report(y_test, y_pred_nb_initial))


Initial Decision Tree Performance:
                           precision    recall  f1-score   support

            cancel_order       0.99      0.95      0.97       187
            change_order       0.93      0.98      0.95       187
 change_shipping_address       0.98      0.97      0.97       216
  check_cancellation_fee       0.99      0.99      0.99       199
           check_invoice       0.96      0.98      0.97       192
   check_payment_methods       0.99      0.99      0.99       206
     check_refund_policy       0.95      0.99      0.97       200
               complaint       1.00      0.99      1.00       203
contact_customer_service       0.99      0.98      0.99       208
     contact_human_agent       0.93      1.00      0.97       201
          create_account       0.98      0.98      0.98       217
          delete_account       0.96      0.96      0.96       178
        delivery_options       0.96      1.00      0.98       218
         delivery_period       0.98   

In [45]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

In [47]:
#parameter grid
param_grid = {
    'max_depth': [5, 10, 15, 20], 
    'min_samples_split': [2, 5, 10], 
    'criterion': ['gini', 'entropy']
}

In [49]:
# Initialize Decision Tree model
dt = DecisionTreeClassifier()

In [51]:
# Performing Grid Search with Cross-Validation
grid_search = GridSearchCV(dt, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [5, 10, 15, 20],
                         'min_samples_split': [2, 5, 10]},
             scoring='accuracy')

In [53]:
# Train model with best parameters
best_dt = grid_search.best_estimator_
y_pred_dt = best_dt.predict(X_test)

In [55]:
# Get Best parameters and accuracy
print("Best Parameters:", grid_search.best_params_)
print("Best Accuracy:", grid_search.best_score_)

Best Parameters: {'criterion': 'entropy', 'max_depth': 20, 'min_samples_split': 2}
Best Accuracy: 0.8218821683788009


#### Hyperparameter Tuning for Naïve Bayes Classifier

In [96]:
#Using the same parameter grid
# Optimizing alpha for Laplace smoothing to avoid zero probabilities.

param_grid = {"alpha": [0.1, 0.5, 1.0, 1.5, 2.0],
}

In [98]:
# Initialize Naïve Bayes model
nb = MultinomialNB()

In [100]:
# Performing Grid Search with Cross-Validation
grid_search = GridSearchCV(nb, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=MultinomialNB(),
             param_grid={'alpha': [0.1, 0.5, 1.0, 1.5, 2.0]},
             scoring='accuracy')

In [101]:
# Best parameters and accuracy
print("Best Parameters:", grid_search.best_params_)
print("Best Accuracy:", grid_search.best_score_)

Best Parameters: {'alpha': 0.1}
Best Accuracy: 0.9873935203968471


In [102]:
# Train model with best parameters
best_nb = grid_search.best_estimator_
y_pred_nb = best_nb.predict(X_test)

In [106]:
#The best alpha value improved classification accuracy.
#The tuned Naïve Bayes model performed better on test data.

In [108]:
y_pred_dt_tuned = best_dt.predict(X_test)
y_pred_nb_tuned = best_nb.predict(X_test)

In [110]:
print("\nTuned Decision Tree Performance:\n", classification_report(y_test, y_pred_dt_tuned))
print("\nTuned Naïve Bayes Performance:\n", classification_report(y_test, y_pred_nb_tuned))


Tuned Decision Tree Performance:
                           precision    recall  f1-score   support

            cancel_order       1.00      0.90      0.95       187
            change_order       0.89      0.97      0.93       187
 change_shipping_address       0.98      0.97      0.98       216
  check_cancellation_fee       0.92      0.56      0.69       199
           check_invoice       0.96      0.92      0.94       192
   check_payment_methods       1.00      0.98      0.99       206
     check_refund_policy       0.79      0.84      0.82       200
               complaint       1.00      0.25      0.40       203
contact_customer_service       0.99      0.95      0.97       208
     contact_human_agent       1.00      0.41      0.58       201
          create_account       0.98      0.91      0.94       217
          delete_account       0.72      0.94      0.82       178
        delivery_options       0.98      0.89      0.94       218
         delivery_period       1.00     

In [114]:
#Accuracy Scores Comparison
initial_dt_acc = accuracy_score(y_test, y_pred_dt_initial)
tuned_dt_acc = accuracy_score(y_test, y_pred_dt_tuned)
initial_nb_acc = accuracy_score(y_test, y_pred_nb_initial)
tuned_nb_acc = accuracy_score(y_test, y_pred_nb_tuned)

In [122]:
print("\nAccuracy Comparison:")
print("Decision Tree: Before = {initial_dt_acc:.2%}, After = {tuned_dt_acc:.2%}")
print("Naïve Bayes: Before = {initial_nb_acc:.2%}, After = {tuned_nb_acc:.2%}")



Accuracy Comparison:
Decision Tree: Before = {initial_dt_acc:.2%}, After = {tuned_dt_acc:.2%}
Naïve Bayes: Before = {initial_nb_acc:.2%}, After = {tuned_nb_acc:.2%}


In [126]:
if tuned_dt_acc > initial_dt_acc:
    print("\nDecision Tree improved after tuning by", round((tuned_dt_acc - initial_dt_acc) * 100, 2), "%")

if tuned_nb_acc > initial_nb_acc:
    print("\nNaïve Bayes improved after tuning by", round((tuned_nb_acc - initial_nb_acc) * 100, 2), "%")
    print("\nHyperparameter tuning successfully improved model performance!")



Naïve Bayes improved after tuning by 0.09 %

Hyperparameter tuning successfully improved model performance!
